# Finding and fixing problems with the Dataset

#### Importing Libraries

In [ ]:
import pandas as pd

#### Importing Data 
here we import the three files that make the main dataset (one for each month jan, feb, march), and merge them into one dataframe (td (tripData))
we also import the metadata for the zones and create a dictionary called zones

In [111]:
jan = pd.read_parquet(r"Data\yellow_tripdata_2025-01.parquet")
feb = pd.read_parquet(r"Data\yellow_tripdata_2025-02.parquet")
mar = pd.read_parquet(r"Data\yellow_tripdata_2025-03.parquet")
zones = pd.read_csv(r"Data\NYC_Taxi_Zones.csv")

td = pd.concat([jan, feb, mar], ignore_index=True)

## Now we will go field by field Finding all the issues with the dataset
### Lets Start with the vendor id 
All entries in vendor id should be either 1, 2, 6, 7

##### TLDR: 
+ This field is perfectly good and contains no issues

In [112]:
# Checking that all Vendor IDS conform to allowed vendor IDs
nonConformingRows = (~td["VendorID"].isin([1,2,6,7])).sum()

#Checking for NA rows
NARows = print(td["VendorID"].isna().sum())

print("Total non conforming rows: ", nonConformingRows)
print("Total NA rows: ", NARows)

0
Total non conforming rows:  0
Total NA rows:  None


### Lets go over the Date fields now
+ The Dates should be in the range jan through march 2025 
+ The dropoff should be after the pickup
+ No NA's are acceptable

##### TLDR:
+ No NA values
+ Plenty of invalidity issues with start and end time 
+ plenty of trips with the same pickup and dropoff time and plenty more where dropoff is before pickup

In [113]:
#first lets check for na values in pickup or dropoff
puNA = td.tpep_pickup_datetime.isna().sum()
doNA = td.tpep_dropoff_datetime.isna().sum()
totNA = ((td.tpep_pickup_datetime.isna()) | (td.tpep_dropoff_datetime.isna())).sum()

#Lets check that all dates are within the valid ranges:
puIV = ((td["tpep_pickup_datetime"] < "2025-01-01") | (td["tpep_pickup_datetime"] >= "2025-04-01")).sum()
doIV = ((td["tpep_dropoff_datetime"] < "2025-01-01") | (td["tpep_dropoff_datetime"] >= "2025-04-01")).sum()
totIV = (((td["tpep_pickup_datetime"] < "2025-01-01") | (td["tpep_pickup_datetime"] >= "2025-04-01")) | ((td["tpep_dropoff_datetime"] < "2025-01-01") | (td["tpep_dropoff_datetime"] > "2025-04-01"))).sum()

#Lets check for validity of dropoff time against pickup time
DObefPU = (td["tpep_pickup_datetime"] >= td["tpep_dropoff_datetime"]).sum()

print("NA pickup time: ", puNA)
print("NA dropoff time: ", doNA)
print("NA pickup or dropoff time or both: ", totNA, "\n")

print("Invalid pickup time: ", puIV)
print("Invalid dropoff time: ", doIV)
print("Invalid pickup or dropoff or both: ", totIV, "\n")

print("Invalid pickup time as compared to dropoff time: ", DObefPU)

print("Invalid times in general sample: ")
display(td[((td["tpep_pickup_datetime"] < "2025-01-01") | (td["tpep_pickup_datetime"] > "2025-04-01")) | ((td["tpep_dropoff_datetime"] < "2025-01-01") | (td["tpep_dropoff_datetime"] > "2025-04-01"))].head())

print("Invalid pickup as compared to dropoff sample: ")
display(td[(td["tpep_pickup_datetime"] >= td["tpep_dropoff_datetime"])].head())

NA pickup time:  0
NA dropoff time:  0
NA pickup or dropoff time or both:  0 

Invalid pickup time:  25
Invalid dropoff time:  993
Invalid pickup or dropoff or both:  993 

Invalid pickup time as compared to dropoff time:  29495
Invalid times in general sample: 


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
605,2,2024-12-31 23:30:03,2024-12-31 23:43:02,1.0,3.00,1.0,N,246,13,1,16.3,1.0,0.5,5.32,0.0,1.0,26.62,2.5,0.0,0.0
687,2,2024-12-31 23:31:38,2024-12-31 23:41:48,1.0,1.03,1.0,N,43,140,2,10.7,1.0,0.5,0.00,0.0,1.0,15.70,2.5,0.0,0.0
688,2,2024-12-31 23:46:38,2025-01-01 00:03:03,1.0,3.95,1.0,N,229,24,2,19.1,1.0,0.5,0.00,0.0,1.0,24.10,2.5,0.0,0.0
861,2,2024-12-31 23:56:19,2025-01-01 00:11:19,6.0,2.28,1.0,N,68,107,1,14.9,1.0,0.5,3.98,0.0,1.0,23.88,2.5,0.0,0.0
1108,2,2024-12-31 23:55:37,2025-01-01 00:01:26,1.0,1.12,1.0,N,56,56,2,7.9,1.0,0.5,0.00,0.0,1.0,10.40,0.0,0.0,0.0


Invalid pickup as compared to dropoff sample: 


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
92,1,2025-01-01 00:49:48,2025-01-01 00:49:48,1.0,0.0,1.0,Y,87,264,2,20.06,0.0,0.0,0.0,0.0,0.0,20.06,0.0,0.0,0.0
11604,1,2025-01-01 01:42:36,2025-01-01 01:42:36,1.0,0.0,1.0,N,158,264,2,3.00,3.5,0.5,0.0,0.0,1.0,8.00,2.5,0.0,0.0
13820,1,2025-01-01 02:13:25,2025-01-01 02:13:25,1.0,0.0,5.0,Y,233,264,2,114.00,0.0,0.0,0.0,0.0,0.0,114.00,0.0,0.0,0.0
14605,1,2025-01-01 02:09:52,2025-01-01 02:09:52,1.0,0.0,1.0,N,237,264,2,3.00,3.5,0.5,0.0,0.0,1.0,8.00,2.5,0.0,0.0
14607,1,2025-01-01 02:49:40,2025-01-01 02:49:40,1.0,0.0,1.0,N,162,264,2,3.00,3.5,0.5,0.0,0.0,1.0,8.00,2.5,0.0,0.0


### Lets go over the Passenger count field now
+ Should not be 0 
+ Should not be NA
+ Needs to be realistic amount (cannot fit more than lets say 7 passengers in a car)
+ cant have negative passengers

##### TLDR:
+ plenty of NA rows
+ plenty of Zero passenger trips
+ plenty of trips with too many passengers
+ no negative passengers

In [115]:
#first lets check for missing values:
passNA = td.passenger_count.isna().sum()

#Now lets check for 0 values (how can a trip with no passengers exist??):
zeroPass = (td["passenger_count"] == 0).sum()

#Now lets check for passenger values that are unrealistic:
tooManyPassengers = (td["passenger_count"] >= 7).sum()

#Now lets check for negative passengers:
negativePassengers = (td["passenger_count"] < 0).sum()

print("rows with NA passengers: ", passNA)
print("rows with 0 passengers: ", zeroPass),
print("rows with unrealistic no. of passengers: ", tooManyPassengers)
print("rows with negative passengers: ", negativePassengers, "\n")

print("Sample of rows with 0 passengers")
display(td[td["passenger_count"] == 0].head())

print("\n", "Sample of rows with too many passengers")
display(td[td["passenger_count"] >= 7].head())

rows with NA passengers:  2263749
rows with 0 passengers:  69122
rows with unrealistic no. of passengers:  47
rows with negative passengers:  0 

Sample of rows with 0 passengers


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
6,1,2025-01-01 00:14:47,2025-01-01 00:16:15,0.0,0.4,1.0,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0
7,1,2025-01-01 00:39:27,2025-01-01 00:51:51,0.0,1.6,1.0,N,234,148,1,12.1,3.5,0.5,2.00,0.0,1.0,19.10,2.5,0.0,0.0
8,1,2025-01-01 00:53:43,2025-01-01 01:13:23,0.0,2.8,1.0,N,148,170,1,19.1,3.5,0.5,3.00,0.0,1.0,27.10,2.5,0.0,0.0
94,1,2025-01-01 00:11:27,2025-01-01 00:16:58,0.0,0.7,1.0,N,144,211,1,7.2,3.5,0.5,0.00,0.0,1.0,12.20,2.5,0.0,0.0
95,1,2025-01-01 00:19:30,2025-01-01 00:27:25,0.0,1.0,1.0,N,211,158,1,9.3,3.5,0.5,2.85,0.0,1.0,17.15,2.5,0.0,0.0



 Sample of rows with too many passengers


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
98,2,2025-01-01 00:04:29,2025-01-01 00:55:58,9.0,31.97,5.0,N,132,265,2,90.0,0.0,0.0,0.00,20.32,1.0,111.32,0.0,0.00,0.0
21363,2,2025-01-01 04:42:17,2025-01-01 04:42:19,8.0,0.00,5.0,N,264,264,1,85.0,0.0,0.0,0.00,0.00,1.0,86.00,0.0,0.00,0.0
78231,2,2025-01-02 07:44:48,2025-01-02 08:02:07,9.0,8.19,5.0,N,261,265,1,90.0,0.0,0.0,10.00,0.00,1.0,101.00,0.0,0.00,0.0
132836,2,2025-01-02 17:56:50,2025-01-02 17:57:15,7.0,0.00,5.0,N,138,138,2,75.0,5.0,0.0,0.00,0.00,1.0,82.75,0.0,1.75,0.0
293430,2,2025-01-04 16:35:35,2025-01-04 16:35:57,8.0,0.04,5.0,N,132,132,1,80.0,0.0,0.0,16.55,0.00,1.0,99.30,0.0,1.75,0.0


### Lets go over the Trip Distance count field now
+ Cannot be negative
+ Cannot be 0
+ Cannot be NA
+ Cannot be unrealistically large

##### TLDR:
+ Many trips with 0 trip distance
+ Everything else is fine

In [116]:
#Checking for negative trip distnaces:
negTD = (td["trip_distance"] < 0).sum()

#Checking for 0 trip distances:
zeroTD = (td["trip_distance"] == 0).sum()

#Checking for NA trip distances:
naTD = td.trip_distance.isna().sum()

#Checking for unrealistically large trip distances:
urlTD = (td["trip_distance"] == 80).sum()

print("rows with NA trip distance: ", naTD)
print("rows with 0 trip distance: ", zeroTD),
print("rows with unrealistic trip distance: ", urlTD)
print("rows with negative trip distance: ", negTD, "\n")

print("sample of rows with 0 trip distance:")
display(td[td["trip_distance"] == 0].head())


rows with NA trip distance:  0
rows with 0 trip distance:  294386
rows with unrealistic trip distance:  0
rows with negative trip distance:  0 

sample of rows with 0 trip distance:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
92,1,2025-01-01 00:49:48,2025-01-01 00:49:48,1.0,0.0,1.0,Y,87,264,2,20.06,0.0,0.0,0.00,0.00,0.0,20.06,0.0,0.0,0.0
204,2,2025-01-01 00:37:43,2025-01-01 00:37:53,1.0,0.0,5.0,N,148,148,1,12.00,0.0,0.0,2.00,0.00,1.0,17.50,2.5,0.0,0.0
358,2,2025-01-01 00:57:08,2025-01-01 00:57:16,3.0,0.0,5.0,N,141,141,1,30.00,0.0,0.0,0.00,0.00,1.0,33.50,2.5,0.0,0.0
505,1,2025-01-01 00:27:40,2025-01-01 00:59:30,1.0,0.0,1.0,N,168,76,1,50.50,0.0,0.5,0.00,6.94,1.0,58.94,0.0,0.0,0.0
619,2,2025-01-01 00:56:49,2025-01-01 00:56:54,4.0,0.0,5.0,N,164,164,1,20.00,0.0,0.0,7.05,0.00,1.0,30.55,2.5,0.0,0.0


### Lets go over the RateCodeID  field now
+ Can only be from set of following valid ones:
    + 1 = Standard rate
    + 2 = JFK
    + 3 = Newark
    + 4 = Nassau or Westchester
    + 5 = Negotiated fare
    + 6 = Group ride
    + 99 = Null/unknown
+ Can not be NA

##### TLDR:
+ no non confirming non NA rows 
+ plenty of missing values

In [117]:
#first lets check it conforms with the IDS that are valid:
nonConformingRID = (~td["RatecodeID"].isin([1,2,3,4,5,6,99])).sum() - td.RatecodeID.isna().sum()

#now lets check for na values:
naRID = td.RatecodeID.isna().sum()

print("number of invalid Rate Code IDS: ", nonConformingRID)
print("number of NA Rate Code IDS: ", naRID)

number of invalid Rate Code IDS:  0
number of NA Rate Code IDS:  2263749


### Lets go over the store and forward flag field now
+ Can only be Y or N
+ Can not be NA

##### TLDR:
+ no non confirming non NA rows 
+ plenty of missing values
+ Generally this field should not affect our work but its good to check

In [118]:
#Check for non conforming rows:
nonConformingSFF = (~td["store_and_fwd_flag"].isin(["Y","N"])).sum() - td.store_and_fwd_flag.isna().sum()

#Check for NA rows:
naSFF = td.store_and_fwd_flag.isna().sum()

print("number of invalid Store and Forward flags: ", nonConformingRID)
print("number of NA Store and Forward flags: ", naRID)

number of invalid Store and Forward flags:  0
number of NA Store and Forward flags:  2263749


### Lets go over the Location ID fields now
+ should be in the metadata dataframe
+ Can not be NA

##### TLDR:
+ no missing values
+ plenty of invalid values

In [125]:
#First Lets check for missing values:
puLIDNA = td.PULocationID.isna().sum()
doLIDNA = td.DOLocationID.isna().sum()

#Now lets check that they exist in the zones lookup dataframe:
lids = zones["Location ID"]
nonConformingPULID = (~td["PULocationID"].isin(lids)).sum() - puLIDNA
nonConformingDOLID = (~td["DOLocationID"].isin(lids)).sum() - doLIDNA

print("Missing pickup location ID: ", puNA)
print("Missing drop off location ID: ", doNA)
print("Invalid pick up location ID: ", nonConformingPULID)
print("Invalid drop off location ID: ", nonConformingDOLID)

print("\nSample invalid PULocationID rows:")
display(td[~td["PULocationID"].isin(lids)].head())

print("\nSample invalid DOLocationID rows:")
display(td[~td["DOLocationID"].isin(lids)].head())

Missing pickup location ID: 0
Missing drop off location ID: 0
Invalid pick up location ID: 27561
Invalid drop off location ID: 69810

Sample invalid PULocationID rows:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
128,2,2025-01-01 00:19:33,2025-01-01 00:30:45,1.0,2.80,1.0,N,264,264,1,14.2,1.0,0.5,3.84,0.0,1.0,23.04,2.5,0.0,0.0
129,2,2025-01-01 00:32:34,2025-01-01 00:47:43,1.0,1.96,1.0,N,264,264,2,14.9,1.0,0.5,0.00,0.0,1.0,17.40,0.0,0.0,0.0
130,2,2025-01-01 01:02:16,2025-01-01 01:15:01,1.0,2.17,1.0,N,264,264,1,14.2,1.0,0.5,3.84,0.0,1.0,23.04,2.5,0.0,0.0
389,1,2025-01-01 00:52:29,2025-01-01 00:58:00,2.0,0.90,1.0,N,264,142,1,7.9,3.5,0.5,2.55,0.0,1.0,15.45,2.5,0.0,0.0
565,1,2025-01-01 00:41:49,2025-01-01 00:49:23,5.0,2.00,1.0,N,264,264,1,10.7,1.0,0.5,3.14,2.5,1.0,18.84,0.0,0.0,0.0



Sample invalid DOLocationID rows:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
92,1,2025-01-01 00:49:48,2025-01-01 00:49:48,1.0,0.00,1.0,Y,87,264,2,20.06,0.0,0.0,0.00,0.00,0.0,20.06,0.0,0.0,0.0
98,2,2025-01-01 00:04:29,2025-01-01 00:55:58,9.0,31.97,5.0,N,132,265,2,90.00,0.0,0.0,0.00,20.32,1.0,111.32,0.0,0.0,0.0
128,2,2025-01-01 00:19:33,2025-01-01 00:30:45,1.0,2.80,1.0,N,264,264,1,14.20,1.0,0.5,3.84,0.00,1.0,23.04,2.5,0.0,0.0
129,2,2025-01-01 00:32:34,2025-01-01 00:47:43,1.0,1.96,1.0,N,264,264,2,14.90,1.0,0.5,0.00,0.00,1.0,17.40,0.0,0.0,0.0
130,2,2025-01-01 01:02:16,2025-01-01 01:15:01,1.0,2.17,1.0,N,264,264,1,14.20,1.0,0.5,3.84,0.00,1.0,23.04,2.5,0.0,0.0


### Lets go over the Payment Type field now
+ should be in the following valid range
    + 0 = Flex Fare trip
    + 1 = Credit card
    + 2 = Cash
    + 3 = No charge
    + 4 = Dispute
    + 5 = Unknown
    + 6 = Voided trip
+ Can not be NA

##### TLDR:
+ no missing values
+ no invalid values

In [126]:
#Check for missing values:
ptNA = td.payment_type.isna().sum()

#Check for value conformity:
nonConformingPT = (~td["payment_type"].isin([0,1,2,3,4,5,6])).sum() - ptNA

print("number of invalid payment type IDS: ", nonConformingPT)
print("number of NA payment types: ", ptNA)

number of invalid payment type IDS:  0
number of NA payment types:  0


### Lets go over the fare amount field now
+ should not be negative
+ Can not be NA

##### TLDR:
+ no missing values
+ plenty of invalid values

In [130]:
#Check for NA values:
fareNA = td.fare_amount.isna().sum()

#Check for Negative Values:
fareIV = (td["fare_amount"] < 0).sum()

print("number of invalid fare amounts: ", fareIV)
print("number of NA fare amounts: ", fareNA, "\n")

print("sample of invalid taxi fares:")
display(td[td["fare_amount"] < 0].head())

number of invalid fare amounts:  535497
number of NA fare amounts:  0 

sample of invalid taxi fares:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
17,2,2025-01-01 00:01:41,2025-01-01 00:07:14,1.0,0.71,1.0,N,79,107,2,-7.2,-1.0,-0.5,3.66,0.0,-1.0,-8.54,-2.5,0.0,0.0
22,2,2025-01-01 00:55:54,2025-01-01 01:00:38,1.0,0.69,1.0,N,137,233,4,-6.5,-1.0,-0.5,0.00,0.0,-1.0,-11.50,-2.5,0.0,0.0
104,2,2025-01-01 00:56:12,2025-01-01 01:15:00,1.0,0.97,1.0,N,161,170,4,-16.3,-1.0,-0.5,0.00,0.0,-1.0,-21.30,-2.5,0.0,0.0
149,2,2025-01-01 00:55:53,2025-01-01 01:06:49,1.0,1.42,1.0,N,79,45,2,-12.1,-1.0,-0.5,0.00,0.0,-1.0,-17.10,-2.5,0.0,0.0
202,2,2025-01-01 00:29:35,2025-01-01 00:36:02,1.0,0.60,1.0,N,79,148,4,-7.2,-1.0,-0.5,0.00,0.0,-1.0,-12.20,-2.5,0.0,0.0


### Lets go over the extra field now
+ should not be negative
+ should not be NA

##### TLDR:
+ no missing values
+ plenty of invalid values

In [134]:
#First lets check for NA values:
extraNA = td.extra.isna().sum()

#Lets check for invalid values:
extraIV = (td["extra"] < 0).sum()

print("number of invalid extra amounts: ", extraIV)
print("number of NA extra amounts: ", extraNA, "\n")

print("sample of invalid values in extra:")
display(td[td["extra"] < 0].head())

number of invalid extra amounts:  92908
number of NA extra amounts:  0 

sample of invalid values in extra:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
17,2,2025-01-01 00:01:41,2025-01-01 00:07:14,1.0,0.71,1.0,N,79,107,2,-7.2,-1.0,-0.5,3.66,0.0,-1.0,-8.54,-2.5,0.0,0.0
22,2,2025-01-01 00:55:54,2025-01-01 01:00:38,1.0,0.69,1.0,N,137,233,4,-6.5,-1.0,-0.5,0.00,0.0,-1.0,-11.50,-2.5,0.0,0.0
104,2,2025-01-01 00:56:12,2025-01-01 01:15:00,1.0,0.97,1.0,N,161,170,4,-16.3,-1.0,-0.5,0.00,0.0,-1.0,-21.30,-2.5,0.0,0.0
149,2,2025-01-01 00:55:53,2025-01-01 01:06:49,1.0,1.42,1.0,N,79,45,2,-12.1,-1.0,-0.5,0.00,0.0,-1.0,-17.10,-2.5,0.0,0.0
202,2,2025-01-01 00:29:35,2025-01-01 00:36:02,1.0,0.60,1.0,N,79,148,4,-7.2,-1.0,-0.5,0.00,0.0,-1.0,-12.20,-2.5,0.0,0.0


### Lets go over the mta tax amount field now
+ should not be negative
+ Can not be NA

##### TLDR:
+ no missing values
+ plenty of invalid values

In [135]:
#First lets check for NA values:
mtaTaxNA = td.mta_tax.isna().sum()

#Lets check for invalid values:
mtaTaxIV = (td["mta_tax"] < 0).sum()

print("number of invalid mta tax amounts: ", mtaTaxIV)
print("number of NA mta tax amounts: ", mtaTaxNA, "\n")

print("sample of invalid values in mta_tax:")
display(td[td["mta_tax"] < 0].head())

number of invalid mta tax amounts:  175826
number of NA mta tax amounts:  0 

sample of invalid values in mta_tax:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
17,2,2025-01-01 00:01:41,2025-01-01 00:07:14,1.0,0.71,1.0,N,79,107,2,-7.2,-1.0,-0.5,3.66,0.0,-1.0,-8.54,-2.5,0.0,0.0
22,2,2025-01-01 00:55:54,2025-01-01 01:00:38,1.0,0.69,1.0,N,137,233,4,-6.5,-1.0,-0.5,0.00,0.0,-1.0,-11.50,-2.5,0.0,0.0
104,2,2025-01-01 00:56:12,2025-01-01 01:15:00,1.0,0.97,1.0,N,161,170,4,-16.3,-1.0,-0.5,0.00,0.0,-1.0,-21.30,-2.5,0.0,0.0
149,2,2025-01-01 00:55:53,2025-01-01 01:06:49,1.0,1.42,1.0,N,79,45,2,-12.1,-1.0,-0.5,0.00,0.0,-1.0,-17.10,-2.5,0.0,0.0
202,2,2025-01-01 00:29:35,2025-01-01 00:36:02,1.0,0.60,1.0,N,79,148,4,-7.2,-1.0,-0.5,0.00,0.0,-1.0,-12.20,-2.5,0.0,0.0


### Lets go over the tip amount field now
+ should not be negative
+ should not exist unless transactions is credit card

##### TLDR:
+ plenty of invalid values

In [137]:
#Lets check for invalid values:
tipIV = ((td["tip_amount"] < 0) | ((td["payment_type"] != 1) & (td["tip_amount"] > 0))).sum()

print("Invalid tips: ", tipIV)

display(td[(td["tip_amount"] < 0) | ((td["payment_type"] != 1) & (td["tip_amount"] > 0))].head())

Invalid tips:  193924


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
17,2,2025-01-01 00:01:41,2025-01-01 00:07:14,1.0,0.71,1.0,N,79,107,2,-7.2,-1.0,-0.5,3.66,0.0,-1.0,-8.54,-2.5,0.0,0.0
2459,2,2025-01-01 00:11:51,2025-01-01 00:28:06,1.0,2.39,1.0,N,137,158,4,-16.3,-1.0,-0.5,4.00,0.0,-1.0,-17.30,-2.5,0.0,0.0
2794,2,2025-01-01 00:54:33,2025-01-01 01:23:24,1.0,2.59,1.0,N,79,246,4,-24.7,-1.0,-0.5,-3.00,0.0,-1.0,-32.70,-2.5,0.0,0.0
2795,2,2025-01-01 00:54:33,2025-01-01 01:23:24,1.0,2.59,1.0,N,79,246,4,24.7,1.0,0.5,3.00,0.0,1.0,32.70,2.5,0.0,0.0
4659,2,2025-01-01 00:54:26,2025-01-01 00:59:05,1.0,0.37,1.0,N,234,107,3,-5.8,-1.0,-0.5,2.16,0.0,-1.0,-8.64,-2.5,0.0,0.0


### Lets go over the congestion surcharge field now
+ should not be negative
+ Can not be NA

##### TLDR:
+ plenty of missing values
+ plenty of invalid values

In [142]:
#first lets check for na values:
csNA = td.congestion_surcharge.isna().sum()

#lets check for validity:
csIV = (td["congestion_surcharge"] < 0).sum()

print("number of invalid congestion surcharges: ", csIV)
print("number of NA congestion surcharges: ", csNA, "\n")

print("sample of invalid values in congestion surcharge:")
display(td[td["congestion_surcharge"] < 0].head())

number of invalid congestion surcharges:  148848
number of NA congestion surcharges:  2263749 

sample of invalid values in congestion surcharge:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
17,2,2025-01-01 00:01:41,2025-01-01 00:07:14,1.0,0.71,1.0,N,79,107,2,-7.2,-1.0,-0.5,3.66,0.0,-1.0,-8.54,-2.5,0.0,0.0
22,2,2025-01-01 00:55:54,2025-01-01 01:00:38,1.0,0.69,1.0,N,137,233,4,-6.5,-1.0,-0.5,0.00,0.0,-1.0,-11.50,-2.5,0.0,0.0
104,2,2025-01-01 00:56:12,2025-01-01 01:15:00,1.0,0.97,1.0,N,161,170,4,-16.3,-1.0,-0.5,0.00,0.0,-1.0,-21.30,-2.5,0.0,0.0
149,2,2025-01-01 00:55:53,2025-01-01 01:06:49,1.0,1.42,1.0,N,79,45,2,-12.1,-1.0,-0.5,0.00,0.0,-1.0,-17.10,-2.5,0.0,0.0
202,2,2025-01-01 00:29:35,2025-01-01 00:36:02,1.0,0.60,1.0,N,79,148,4,-7.2,-1.0,-0.5,0.00,0.0,-1.0,-12.20,-2.5,0.0,0.0


### Lets go over the total amount field now
+ should not be negative
+ Can not be NA
+ should be equal to everything added up

##### TLDR:
+ no missing values
+ plenty of invalid values

In [ ]:
#first lets check for na values
totalNA = td.total_amount.isna().sum()

#lets check for invalid values
calc_total = (
    td["fare_amount"] +
    td["extra"] +
    td["mta_tax"] +
    td["tip_amount"] +
    td["tolls_amount"] +
    td["improvement_surcharge"] +
    td["congestion_surcharge"] +
    td["Airport_fee"] +
    td["cbd_congestion_fee"]
)

totalIV = ((abs(td["total_amount"] - calc_total) > 0.01) | (td["total_amount"] < 0)).sum()

print("number of invalid total amounts: ", totalIV)
print("number of NA total amounts: ", totalNA, "\n")

print("sample of invalid values in total amounts:")
display(td[(abs(td["total_amount"] - calc_total) > 0.01) | (td["total_amount"] < 0)].head())

number of invalid total amounts:  2044394
number of NA total amounts:  0 

sample of invalid values in mta_tax:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.6,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.5,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.6,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
6,1,2025-01-01 00:14:47,2025-01-01 00:16:15,0.0,0.4,1.0,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0
7,1,2025-01-01 00:39:27,2025-01-01 00:51:51,0.0,1.6,1.0,N,234,148,1,12.1,3.5,0.5,2.00,0.0,1.0,19.10,2.5,0.0,0.0
